# AMP-CVAE Kaggle Notebook
This notebook clones the AMP-CVAE repository, installs dependencies, and runs training and generation of novel Antimicrobial Peptides.

**⚠️ IMPORTANT GPU NOTE:** If you are running this on Kaggle, please ensure your Accelerator is set to **GPU T4x2** instead of **GPU P100**. The latest versions of PyTorch no longer support the older architecture of the P100 GPU (sm_60), which will result in a CUDA error: no kernel image is available. The T4 GPUs are fully supported.

In [ ]:
!git clone https://github.com/Phan-Trung-Thuan/amp-cvae.git
%cd amp-cvae
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.append('src')
import torch
from dataset import get_dataloaders
from train import train_cvae
from generate import generate_peptides, predict_properties
import os

In [ ]:
# Assuming aps_database.csv is uploaded to Kaggle input or already in the repo
csv_path = 'data/aps_database.csv'

# Train the model
model, vocab, struct_enc, act_bin = train_cvae(csv_path, epochs=10, batch_size=32, seq_type='lstm', lr=1e-3)

In [ ]:
# Generate new peptides
target_charge = 3.0
target_hydro = 45.0
target_struct_idx = 1 # E.g., Alpha-helix
target_activity_vec = [1.0] * 27 # Example vector for activities

generated_seqs = generate_peptides(
    model, 
    vocab, 
    target_charge=target_charge, 
    target_hydro=target_hydro, 
    target_struct_idx=target_struct_idx, 
    target_activity_vec=target_activity_vec, 
    num_samples=5, 
    max_len=50
)

print("Generated Peptides:")
for i, seq in enumerate(generated_seqs):
    print(f"{i+1}: {seq}")

print("\n--- Verifying Properties with Encoder ---")
predictions = predict_properties(model, vocab, generated_seqs, struct_enc, act_bin)
for p in predictions:
    print(f"\nSequence: {p['sequence']}")
    print(f"  Predicted Charge: {p['charge']:.2f} (Target: {target_charge})")
    print(f"  Predicted Hydrophobicity: {p['hydrophobicity']:.2f} (Target: {target_hydro})")
    print(f"  Predicted Structure: {p['structure']}")
    print(f"  Predicted Activities: {p['activities']}")
